# 🏥 Hospital Readmission Predictor for Chronic Condition Patients## Ethical AI Healthcare Application for Singapore## Ethical AI Healthcare Application### Project OverviewThis project develops a machine learning model to predict 30-day hospital readmissions for patients with chronic conditions, adapted for the Singapore healthcare context. While using diabetes patient data as a proxy (due to its prevalence and well-documented nature), the framework is designed to generalize across multiple chronic conditions affecting Singapore aging population., adapted for the Singapore healthcare context.### Dataset Source**UCI Diabetes 130-US Hospitals Dataset (1999-2008)**- URL: https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008### Singapore Healthcare Relevance- High diabetes prevalence (~11% of adults)- Aging population challenges- Healthcare efficiency goals- Preventive care initiatives### Singapore Healthcare Context**Demographic Challenges:**- **Aging Population**: By 2030, 1 in 4 Singaporeans will be aged 65+- **Chronic Disease Burden**:   - 11% of adults have diabetes  - 40% have hypertension    - 38% have high cholesterol- **MOH Health 2020 Goals**: Reduce avoidable hospital readmissions by 15%- **Smart Nation Initiative**: Leverage AI for predictive healthcare analytics**Impact on Singaporeans:**- Early identification of high-risk chronic disease patients- Reduced hospital burden and healthcare costs (~SGD $5,000-10,000 per prevented readmission)- Improved patient care through proactive intervention- Better resource allocation for public hospitals (SGH, NUH, TTSH)**Dataset Note:**This project uses the UCI Diabetes dataset as a proxy for chronic condition management. Diabetes is one of the most prevalent chronic conditions in Singapore, and its clinical features (medications, lab procedures, length of stay, comorbidities) are representative of broader chronic disease patterns. The framework developed here can be extended to other chronic conditions with appropriate data.### Ethical Considerations**Data Privacy & Security:**- Patient data must be handled in compliance with PDPA (Personal Data Protection Act)- De-identification and anonymization protocols required for Singapore hospital data- Secure storage and access controls essential**Bias & Fairness:**- Ensure model performs equitably across different ethnic groups (Chinese, Malay, Indian, Others)- Monitor for age-related bias (elderly patients may have different risk profiles)- Validate on diverse Singapore population before deployment**Responsible AI Use:**- Model predictions should support, not replace, clinical judgment- Clear communication of uncertainty in predictions- Regular auditing for drift and performance degradation- Transparency with patients about AI-assisted care decisions**Alignment with National Goals:**- Supports MOH's Vision 2030 for preventive care- Contributes to Smart Nation health initiatives- Enables data-driven healthcare policy decisions

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('seaborn-v0_8-whitegrid')

import sys
sys.path.append('../src')
from data_loader import DataLoader
from eda import EDAAnalyzer
from preprocessing import DataPreprocessor
from model_training import ModelTrainer
from evaluation import ModelEvaluator

print('✓ All imports successful')

In [ ]:
# Load data
loader = DataLoader()
try:
    df = loader.load_from_csv('../data/raw/diabetes_data_upload.csv')
    loader.display_overview()
except FileNotFoundError:
    print('Dataset not found - using sample data')
    n = 10000
    np.random.seed(42)
    df = pd.DataFrame({
        'age': np.random.choice(['10-20','20-30','30-40','40-50','50-60','60-70','70-80'], n),
        'time_in_hospital': np.random.randint(1, 14, n),
        'num_lab_procedures': np.random.randint(1, 100, n),
        'num_medications': np.random.randint(1, 50, n),
        'number_diagnoses': np.random.randint(1, 10, n),
        'gender': np.random.choice(['Male', 'Female'], n),
        'readmitted': np.random.choice(['NO', 'YES', '<30', '>30'], n, p=[0.5, 0.25, 0.15, 0.10])
    })
print(f'Dataset shape: {df.shape}')

In [ ]:
# EDA
eda = EDAAnalyzer(df, '../results/eda')
eda.generate_full_report(save_figs=True)

In [ ]:
# Preprocessing
prep = DataPreprocessor(test_size=0.2)
X_train, X_test, y_train, y_test = prep.fit_transform(df, 'readmitted', handle_imbalance=True)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train models
trainer = ModelTrainer()
comparison_df = trainer.train_all_models(X_train, y_train, X_test, y_test, tune_hyperparameters=True, cv_folds=5)
print(comparison_df.sort_values('roc_auc', ascending=False))

In [ ]:
# Evaluate best model
best_name, best_model = trainer.select_best_model(comparison_df)
evaluator = ModelEvaluator('../results/evaluation')
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]
metrics = evaluator.calculate_all_metrics(y_test, y_pred, y_proba)
print(f'Metrics: {metrics}')
print(evaluator.interpret_metrics_for_healthcare(metrics, best_name))